# SBBF 3D Mesh Voxel Experiments

This notebook runs Spatial-Blocked Bloom Filter (SBBF) experiments on voxelized 3D meshes.

**Features:**
- Load voxelized meshes (bunny, teapot) at different resolutions
- Configure SBBF parameters (log_num_blocks, hash_k, SFC type)
- Measure false positive rates
- Apply neighbor-based denoising to filter isolated FPs
- Render side-by-side comparisons with PyVista

**Prerequisites:**
- Build the counting_globimap module: `cd build && make counting_globimap`
- Run voxelization: `uv run python experiments/python/voxelize_meshes.py`

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add build directory to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "build"))

import counting_globimap as cg
import numpy as np
import h5py
import pyvista as pv
from tqdm.notebook import tqdm

# Paths
HDF5_DIR = PROJECT_ROOT / "datasets" / "hdf5"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Enable interactive PyVista in notebook
pv.set_jupyter_backend('static')  # Use 'trame' for interactive

print(f"Available SFC types: {[s.name for s in cg.SFCType.__members__.values()]}")
print(f"Available strategies: {[s.name for s in cg.IntraBlockStrategy.__members__.values()]}")

## 2. Helper Functions

In [ ]:
def load_voxels(mesh_name: str, resolution: int) -> np.ndarray:
    """Load voxel coordinates from HDF5 file."""
    path = HDF5_DIR / f"{mesh_name}_{resolution}.h5"
    if not path.exists():
        raise FileNotFoundError(f"Voxel file not found: {path}")
    
    with h5py.File(path, "r") as f:
        coords = f["coords"][:]
        print(f"Loaded {len(coords):,} voxels from {path.name}")
        print(f"  Resolution: {resolution}³ = {resolution**3:,} grid points")
        print(f"  Fill ratio: {len(coords) / resolution**3:.2%}")
    return coords


def list_available_datasets():
    """List available voxelized mesh datasets."""
    datasets = []
    for f in sorted(HDF5_DIR.glob("*_*.h5")):
        parts = f.stem.rsplit("_", 1)
        if len(parts) == 2 and parts[1].isdigit():
            mesh_name, resolution = parts[0], int(parts[1])
            with h5py.File(f, "r") as hf:
                num_voxels = len(hf["coords"])
            datasets.append((mesh_name, resolution, num_voxels, f.name))
    
    print("Available voxelized datasets:")
    print(f"{'Mesh':<15} {'Resolution':<12} {'Voxels':>12} {'File'}")
    print("-" * 60)
    for mesh, res, vox, fname in datasets:
        print(f"{mesh:<15} {res}³{'':<8} {vox:>12,} {fname}")
    return datasets


list_available_datasets()

In [ ]:
def create_sbbf(log_num_blocks: int, hash_k: int, 
                sfc_type: cg.SFCType = cg.SFCType.HILBERT_3D,
                sfc_bits: int = 16) -> cg.SpatialBlockedBloomFilter:
    """Create and configure an SBBF instance."""
    config = cg.SBBFConfig()
    config.sfc_type = sfc_type
    config.log_num_blocks = log_num_blocks
    config.hash_k = hash_k
    config.bits_per_block = 64
    config.sfc_bits = sfc_bits
    
    sbbf = cg.SpatialBlockedBloomFilter(config)
    
    print(f"SBBF Configuration:")
    print(f"  SFC type: {sfc_type.name}")
    print(f"  Blocks: 2^{log_num_blocks} = {2**log_num_blocks:,}")
    print(f"  Hash k: {hash_k}")
    print(f"  Memory: {sbbf.memory_bytes() / 1024:.1f} KB")
    
    return sbbf


def insert_voxels(sbbf: cg.SpatialBlockedBloomFilter, coords: np.ndarray):
    """Insert all voxel coordinates into SBBF."""
    for x, y, z in tqdm(coords, desc="Inserting voxels"):
        sbbf.put3d(int(x), int(y), int(z))
    print(f"Inserted {len(coords):,} voxels, fill ratio: {sbbf.fill_ratio():.4f}")

In [ ]:
def query_all_points(sbbf: cg.SpatialBlockedBloomFilter, 
                     resolution: int, 
                     true_coords: np.ndarray) -> dict:
    """Query all grid points and calculate FPR."""
    true_set = set(map(tuple, true_coords))
    
    queried = []
    false_positives = []
    
    for x in tqdm(range(resolution), desc="Querying grid"):
        for y in range(resolution):
            for z in range(resolution):
                if sbbf.query3d(x, y, z):
                    queried.append((x, y, z))
                    if (x, y, z) not in true_set:
                        false_positives.append((x, y, z))
    
    queried = np.array(queried) if queried else np.empty((0, 3))
    fps = np.array(false_positives) if false_positives else np.empty((0, 3))
    
    num_negatives = resolution**3 - len(true_coords)
    fpr = len(fps) / num_negatives if num_negatives > 0 else 0.0
    
    print(f"\nResults:")
    print(f"  Queried positive: {len(queried):,}")
    print(f"  True positives: {len(queried) - len(fps):,}")
    print(f"  False positives: {len(fps):,}")
    print(f"  FPR: {fpr:.4%}")
    
    return {
        "queried": queried,
        "false_positives": fps,
        "fpr": fpr,
        "true_set": true_set
    }

In [ ]:
def denoise_neighbors(sbbf: cg.SpatialBlockedBloomFilter,
                      candidates: np.ndarray,
                      min_neighbors: int = 2) -> np.ndarray:
    """
    Filter points by requiring minimum neighbor count.
    Real surface voxels have many neighbors; isolated FPs don't.
    """
    if len(candidates) == 0:
        return candidates
    
    filtered = []
    for x, y, z in tqdm(candidates, desc=f"Denoising (min_neighbors={min_neighbors})"):
        neighbor_count = sbbf.neighbors3d(int(x), int(y), int(z), full_26=True)
        if neighbor_count >= min_neighbors:
            filtered.append((x, y, z))
    
    result = np.array(filtered) if filtered else np.empty((0, 3))
    print(f"  Kept {len(result):,} / {len(candidates):,} points ({len(candidates) - len(result):,} removed)")
    return result

In [ ]:
def render_comparison(ground_truth: np.ndarray,
                      raw_query: np.ndarray,
                      false_positives: np.ndarray,
                      denoised: np.ndarray,
                      fpr_raw: float,
                      fpr_denoised: float,
                      title: str = "",
                      save_path: Path = None,
                      point_size: int = 5):
    """
    Render side-by-side comparison:
    - Ground truth (green)
    - Raw SBBF (blue=TP, red=FP)
    - Denoised (cyan)
    """
    plotter = pv.Plotter(shape=(1, 3), window_size=(1800, 600))
    
    # Panel 1: Ground truth
    plotter.subplot(0, 0)
    plotter.add_title(f"Ground Truth\n({len(ground_truth):,} voxels)", font_size=10)
    if len(ground_truth) > 0:
        plotter.add_mesh(pv.PolyData(ground_truth.astype(float)), 
                        color='green', point_size=point_size, render_points_as_spheres=True)
    plotter.add_axes()
    plotter.camera_position = 'iso'
    
    # Panel 2: Raw SBBF
    plotter.subplot(0, 1)
    plotter.add_title(f"Raw SBBF\n(FPR={fpr_raw:.2%})", font_size=10)
    if len(raw_query) > 0:
        fp_set = set(map(tuple, false_positives)) if len(false_positives) > 0 else set()
        tps = np.array([p for p in raw_query if tuple(p) not in fp_set])
        if len(tps) > 0:
            plotter.add_mesh(pv.PolyData(tps.astype(float)), 
                            color='blue', point_size=point_size, render_points_as_spheres=True)
        if len(false_positives) > 0:
            plotter.add_mesh(pv.PolyData(false_positives.astype(float)), 
                            color='red', point_size=point_size+2, render_points_as_spheres=True)
    plotter.add_axes()
    plotter.camera_position = 'iso'
    
    # Panel 3: Denoised
    plotter.subplot(0, 2)
    plotter.add_title(f"Denoised\n(FPR={fpr_denoised:.2%})", font_size=10)
    if len(denoised) > 0:
        plotter.add_mesh(pv.PolyData(denoised.astype(float)), 
                        color='cyan', point_size=point_size, render_points_as_spheres=True)
    plotter.add_axes()
    plotter.camera_position = 'iso'
    
    if save_path:
        plotter.screenshot(str(save_path))
        print(f"Saved: {save_path}")
    
    return plotter.show()

## 3. Run an Experiment

Configure your experiment parameters below:

In [ ]:
# ============================================================
# EXPERIMENT CONFIGURATION - Modify these!
# ============================================================

# Mesh selection
MESH_NAME = "bunny"       # "bunny" or "teapot"
RESOLUTION = 64           # 64, 128, or 256

# SBBF parameters
LOG_NUM_BLOCKS = 14       # 2^14 = 16K blocks = 128 KB
HASH_K = 4                # bits per element
SFC_TYPE = cg.SFCType.HILBERT_3D  # HILBERT_3D or MORTON_3D

# Denoising
MIN_NEIGHBORS = 2         # minimum neighbors to keep a point

# ============================================================

In [ ]:
# Load voxels
coords = load_voxels(MESH_NAME, RESOLUTION)

In [ ]:
# Create and populate SBBF
import math
sfc_bits = max(8, int(math.ceil(math.log2(RESOLUTION + 1))))

sbbf = create_sbbf(
    log_num_blocks=LOG_NUM_BLOCKS,
    hash_k=HASH_K,
    sfc_type=SFC_TYPE,
    sfc_bits=sfc_bits
)

insert_voxels(sbbf, coords)

In [ ]:
# Query all grid points
results = query_all_points(sbbf, RESOLUTION, coords)

In [ ]:
# Apply denoising
denoised = denoise_neighbors(sbbf, results["queried"], min_neighbors=MIN_NEIGHBORS)

# Calculate denoised FPR
fp_after = sum(1 for p in denoised if tuple(p) not in results["true_set"])
num_negatives = RESOLUTION**3 - len(coords)
fpr_denoised = fp_after / num_negatives if num_negatives > 0 else 0.0

print(f"\nDenoised FPR: {fpr_denoised:.4%}")
print(f"FPs removed: {len(results['false_positives']) - fp_after:,}")

In [ ]:
# Render comparison
save_path = FIGURES_DIR / f"{MESH_NAME}_{RESOLUTION}_{SFC_TYPE.name}_k{HASH_K}_b{LOG_NUM_BLOCKS}.png"

render_comparison(
    ground_truth=coords,
    raw_query=results["queried"],
    false_positives=results["false_positives"],
    denoised=denoised,
    fpr_raw=results["fpr"],
    fpr_denoised=fpr_denoised,
    save_path=save_path,
    point_size=max(3, 8 - RESOLUTION // 64)
)

## 4. Parameter Sweep

Compare different SBBF configurations:

In [ ]:
# Parameter sweep configuration
SWEEP_MESH = "bunny"
SWEEP_RESOLUTION = 64

SWEEP_LOG_BLOCKS = [10, 12, 14, 16]  # 2^10=1K to 2^16=64K blocks
SWEEP_HASH_K = [3, 4, 5, 6]
SWEEP_SFC_TYPES = [cg.SFCType.HILBERT_3D, cg.SFCType.MORTON_3D]

In [ ]:
import pandas as pd

# Load voxels once
sweep_coords = load_voxels(SWEEP_MESH, SWEEP_RESOLUTION)
sweep_true_set = set(map(tuple, sweep_coords))

sweep_results = []

for sfc_type in SWEEP_SFC_TYPES:
    for log_blocks in SWEEP_LOG_BLOCKS:
        for hash_k in SWEEP_HASH_K:
            print(f"\nConfig: {sfc_type.name}, blocks=2^{log_blocks}, k={hash_k}")
            
            # Create SBBF
            config = cg.SBBFConfig()
            config.sfc_type = sfc_type
            config.log_num_blocks = log_blocks
            config.hash_k = hash_k
            config.bits_per_block = 64
            config.sfc_bits = 8
            
            sbbf = cg.SpatialBlockedBloomFilter(config)
            
            # Insert
            for x, y, z in sweep_coords:
                sbbf.put3d(int(x), int(y), int(z))
            
            # Count FPs (sample for speed)
            fps = 0
            samples = 0
            for x in range(SWEEP_RESOLUTION):
                for y in range(SWEEP_RESOLUTION):
                    for z in range(SWEEP_RESOLUTION):
                        if (x, y, z) not in sweep_true_set:
                            samples += 1
                            if sbbf.query3d(x, y, z):
                                fps += 1
            
            fpr = fps / samples if samples > 0 else 0
            
            sweep_results.append({
                "sfc_type": sfc_type.name,
                "log_blocks": log_blocks,
                "num_blocks": 2**log_blocks,
                "hash_k": hash_k,
                "memory_kb": sbbf.memory_bytes() / 1024,
                "fill_ratio": sbbf.fill_ratio(),
                "fpr": fpr,
                "fps": fps
            })
            print(f"  FPR: {fpr:.4%}, Memory: {sbbf.memory_bytes()/1024:.1f} KB")

df = pd.DataFrame(sweep_results)
print("\n" + "="*60)
print("Sweep complete!")

In [ ]:
# Display results table
df_display = df.copy()
df_display["fpr"] = df_display["fpr"].apply(lambda x: f"{x:.4%}")
df_display["memory_kb"] = df_display["memory_kb"].apply(lambda x: f"{x:.1f}")
df_display["fill_ratio"] = df_display["fill_ratio"].apply(lambda x: f"{x:.4f}")
df_display

In [ ]:
# Plot FPR vs memory
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for sfc_type in df["sfc_type"].unique():
    subset = df[df["sfc_type"] == sfc_type]
    
    # FPR vs Memory
    axes[0].scatter(subset["memory_kb"], subset["fpr"] * 100, 
                   label=sfc_type, alpha=0.7, s=50)
    
    # FPR vs hash_k (grouped by log_blocks)
    for log_b in subset["log_blocks"].unique():
        sub = subset[subset["log_blocks"] == log_b]
        axes[1].plot(sub["hash_k"], sub["fpr"] * 100, 
                    marker='o', label=f"{sfc_type}, 2^{log_b}")

axes[0].set_xlabel("Memory (KB)")
axes[0].set_ylabel("FPR (%)")
axes[0].set_title("FPR vs Memory")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("hash_k")
axes[1].set_ylabel("FPR (%)")
axes[1].set_title("FPR vs hash_k")
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / f"{SWEEP_MESH}_{SWEEP_RESOLUTION}_sweep.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. Resolution Comparison

Compare how FPR scales with voxel grid resolution:

In [ ]:
# Compare resolutions for a fixed SBBF size
RES_MESH = "bunny"
RES_LOG_BLOCKS = 14
RES_HASH_K = 4
RES_RESOLUTIONS = [64, 128]  # Add 256 if you have time

res_results = []

for resolution in RES_RESOLUTIONS:
    print(f"\n=== Resolution: {resolution}³ ===")
    
    try:
        coords = load_voxels(RES_MESH, resolution)
    except FileNotFoundError:
        print(f"  Skipping - dataset not found")
        continue
    
    true_set = set(map(tuple, coords))
    
    # Create SBBF with fixed config
    config = cg.SBBFConfig()
    config.sfc_type = cg.SFCType.HILBERT_3D
    config.log_num_blocks = RES_LOG_BLOCKS
    config.hash_k = RES_HASH_K
    config.bits_per_block = 64
    config.sfc_bits = max(8, int(math.ceil(math.log2(resolution + 1))))
    
    sbbf = cg.SpatialBlockedBloomFilter(config)
    
    # Insert
    for x, y, z in tqdm(coords, desc="Inserting"):
        sbbf.put3d(int(x), int(y), int(z))
    
    # Full query
    fps = 0
    negatives = 0
    for x in tqdm(range(resolution), desc="Querying"):
        for y in range(resolution):
            for z in range(resolution):
                if (x, y, z) not in true_set:
                    negatives += 1
                    if sbbf.query3d(x, y, z):
                        fps += 1
    
    fpr = fps / negatives if negatives > 0 else 0
    
    res_results.append({
        "resolution": resolution,
        "grid_points": resolution**3,
        "voxels": len(coords),
        "fill_ratio": len(coords) / resolution**3,
        "fpr": fpr,
        "fps": fps,
        "sbbf_fill": sbbf.fill_ratio()
    })
    
    print(f"  FPR: {fpr:.4%}")

res_df = pd.DataFrame(res_results)
res_df

## 6. Interactive Visualization

Visualize a single mesh with PyVista:

In [ ]:
# Single mesh visualization
VIS_MESH = "teapot"
VIS_RESOLUTION = 64

vis_coords = load_voxels(VIS_MESH, VIS_RESOLUTION)

plotter = pv.Plotter()
plotter.add_mesh(pv.PolyData(vis_coords.astype(float)), 
                color='green', point_size=8, render_points_as_spheres=True)
plotter.add_title(f"{VIS_MESH.title()} ({VIS_RESOLUTION}³)")
plotter.add_axes()
plotter.camera_position = 'iso'
plotter.show()